# Train sp1DLC from local Colab storage

Select a **GPU** runtime. Stop the old training cell before running this notebook. Run the code cells in order. If you are in the same Colab session, the prepared ZIP is already under `/content`, so setup reuses it. Images and training data stay on the fast local VM; checkpoints and evaluation results are saved in Google Drive.


In [ ]:
%pip -q install --pre deeplabcut


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
from pathlib import Path
import shutil, zipfile, yaml, torch, deeplabcut

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
MY_DRIVE = Path('/content/drive/MyDrive')
EXPORT = MY_DRIVE / 'sp1DLC' / 'MobileDLC_Colab_Export_2026-09-24'
ZIP = Path('/content/sp1DLC_colab_ready_20260924.zip')
if not ZIP.is_file() or ZIP.stat().st_size != 169245069:
    parts = [EXPORT / f'sp1DLC_colab_ready_20260924.zip.part{i:02d}' for i in range(1, 10)]
    missing = [part.name for part in parts if not part.is_file()]
    assert not missing, f'Missing Drive transfer files: {missing}'
    with ZIP.open('wb') as output:
        for part in parts:
            with part.open('rb') as source:
                shutil.copyfileobj(source, output)
assert ZIP.stat().st_size == 169245069, 'Project transfer was incomplete.'

NAME = 'mobile_fast_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
LOCAL_RUN = Path('/content') / NAME
PROJECT = LOCAL_RUN / 'sp1DLC'
CONFIG = PROJECT / 'config.yaml'
DRIVE_RUN = MY_DRIVE / 'sp1DLC' / 'Colab_Training_Runs' / NAME
with zipfile.ZipFile(ZIP) as archive:
    files = [name for name in archive.namelist() if not name.endswith('/')]
    assert len(files) == 1928 and all(name.startswith('sp1DLC/') and '..' not in Path(name).parts for name in files), 'Unexpected archive contents.'
    assert archive.testzip() is None, 'Project archive failed its integrity check.'
    LOCAL_RUN.mkdir(exist_ok=False)
    archive.extractall(LOCAL_RUN)

with CONFIG.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['project_path'] = str(PROJECT)
config['video_sets'] = {str(PROJECT / 'videos' / Path(str(old).replace('\\', '/')).name): value for old, value in config['video_sets'].items()}
with CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

images = list((PROJECT / 'labeled-data').glob('Test */*.png'))
tables = list((PROJECT / 'labeled-data').glob('Test */CollectedData_Kobe.h5'))
assert len(images) == 1865 and len(tables) == 31, (len(images), len(tables))
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
for name in ('dlc-models-pytorch', 'evaluation-results-pytorch'):
    target = DRIVE_RUN / name
    target.mkdir()
    (PROJECT / name).symlink_to(target, target_is_directory=True)
shutil.copy2(CONFIG, DRIVE_RUN / 'config.yaml')
print('Local training project:', PROJECT)
print('Persistent Drive outputs:', DRIVE_RUN)
print('GPU:', torch.cuda.get_device_name(0))
print('Verified', len(images), 'images and', len(tables), 'label tables')


In [ ]:
assert not list((DRIVE_RUN / 'dlc-models-pytorch').rglob('snapshot-*.pt')), 'A checkpoint already exists in this run. Resume instead of restarting.'
if not (PROJECT / 'training-datasets').exists():
    deeplabcut.create_training_dataset(str(CONFIG), Shuffles=[1], net_type='resnet_50', engine=deeplabcut.Engine.PYTORCH)
    shutil.copytree(PROJECT / 'training-datasets', DRIVE_RUN / 'training-datasets')
deeplabcut.train_network(str(CONFIG), shuffle=1, epochs=200, batch_size=8, save_epochs=5)
print('Training finished. Checkpoints:', DRIVE_RUN / 'dlc-models-pytorch')


In [ ]:
assert list((DRIVE_RUN / 'dlc-models-pytorch').rglob('snapshot-*.pt')), 'No trained checkpoint found yet.'
deeplabcut.evaluate_network(str(CONFIG), snapshotindex=-1, per_keypoint_evaluation=True, plotting=False)
results = sorted((DRIVE_RUN / 'evaluation-results-pytorch').rglob('CombinedEvaluation-results.csv'))
print('Evaluation saved in:', DRIVE_RUN / 'evaluation-results-pytorch')
if results:
    import pandas as pd
    display(pd.read_csv(results[-1]))


The images in `/content` disappear when this Colab runtime ends. The original transfer files, copied config and training dataset, checkpoints, and evaluation results remain in Drive. Do not rerun the setup cell while training is active. If the runtime disconnects, use the saved checkpoint to resume rather than starting another run.
